# Load improved model

Choose a saved improved run, load its checkpoint, verify save/load behavior, plot training curves, and inspect decoded validation predictions.


In [1]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import tempfile
from pathlib import Path

import keras
import matplotlib.pyplot as plt
import numpy as np

from src.data.freihand import FreiHand, SPLIT_SEED, SPLIT_VALIDATION_FRACTION
from src.evaluation.metrics import evaluate_model, sample_mpke
from src.evaluation.overlays import prediction_grid, save_figure
from src.evaluation.training_curves import load_history, plot_training_curves

from src.models.heatmaps import wrap_with_keypoint_decoder


## Choose run


In [2]:
IMPROVED_RUN = 'improved-model-1'
AVAILABLE_IMPROVED = ('improved-model-1', 'improved-model-1-online-augmented')

if IMPROVED_RUN not in AVAILABLE_IMPROVED:
    raise ValueError(f'Unknown improved run: {IMPROVED_RUN}')

run_dir = project_root / 'models' / IMPROVED_RUN
checkpoint_path = run_dir / 'best.keras'
config_path = project_root / 'logs' / IMPROVED_RUN / 'config.json'
evaluation_path = project_root / 'artifacts' / IMPROVED_RUN / 'evaluation.json'

if not config_path.exists():
    raise FileNotFoundError(f'Missing run config: {config_path}')
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')

config = json.loads(config_path.read_text())
input_size = int(config.get('input_shape', [224])[0])
print(f'Run:        {IMPROVED_RUN}')
print(f'Model ID:   {config.get("model_id", config.get("model"))}')
print(f'Checkpoint: {checkpoint_path}')
print(f'Input size: {input_size}')
print(json.dumps(config, indent=2))


Run:        improved-model-1
Model ID:   improved-model-1
Checkpoint: /Users/victor/Projects/code/hand-pose-estimation/models/improved-model-1/best.keras
Input size: 224
{
  "run_name": "improved-model-1",
  "model_id": "improved-model-1",
  "model": "residual_heatmap_cnn",
  "epochs": 30,
  "batch_size": 32,
  "learning_rate": 0.001,
  "validation_fraction": 0.1,
  "seed": 42,
  "n_train": 117216,
  "n_val": 3256,
  "n_train_base_samples": 29304,
  "n_val_base_samples": 3256,
  "train_variants": [
    "gs",
    "hom",
    "sample",
    "auto"
  ],
  "val_variants": [
    "gs"
  ],
  "input_shape": [
    224,
    224,
    3
  ],
  "representation": "heatmap",
  "heatmap_size": 56,
  "heatmap_sigma": 2.0,
  "loss": "mse",
  "metrics": [
    "mae"
  ],
  "optimizer": "adam"
}


## Training curves


In [3]:
history = load_history(IMPROVED_RUN)
fig = plot_training_curves(history, suptitle=f'Improved training: {IMPROVED_RUN}')
saved_path = save_figure(fig, f'{IMPROVED_RUN}_training_curves')
print(f'Saved figure: {saved_path.relative_to(project_root)}')
plt.show()


Saved figure: reports/figures/improved-model-1_training_curves.png


/var/folders/6x/nj8_bs6j4yq01x8v0rvq1gdr0000gn/T/ipykernel_65886/3487966887.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Load and verify checkpoint


In [4]:
model = keras.models.load_model(checkpoint_path)
model.summary()

dummy = np.random.default_rng(0).random((4, input_size, input_size, 3), dtype=np.float32)
expected = model.predict(dummy, verbose=0)

with tempfile.TemporaryDirectory() as tmp:
    roundtrip_path = Path(tmp) / 'roundtrip.keras'
    model.save(roundtrip_path)
    reloaded = keras.models.load_model(roundtrip_path)
    actual = reloaded.predict(dummy, verbose=0)

assert np.array_equal(expected, actual), 'Round-trip predictions do not match'
print('Round-trip OK: reloaded model produces identical raw outputs.')

eval_model = wrap_with_keypoint_decoder(model, input_size=input_size) if len(model.output_shape) == 4 else model


Model: "residual_heatmap_cnn"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ input_image[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_relu (ReLU)    │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_conv1 │ (None, 112, 112,  │      9,216 │ stem_relu[0][0]   │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_bn1   │ (None, 112, 112,  │        128 │ stage1_block1_co… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_relu1 │ (None, 112, 112,  │          0 │ stage1_block1_bn… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_conv2 │ (None, 112, 112,  │      9,216 │ stage1_block1_re… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_bn2   │ (None, 112, 112,  │        128 │ stage1_block1_co… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_add   │ (None, 112, 112,  │          0 │ stage1_block1_bn… │
│ (Add)               │ 32)               │            │ stem_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_block1_relu2 │ (None, 112, 112,  │          0 │ stage1_block1_ad… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_block1_conv1 │ (None, 56, 56,    │     18,432 │ stage1_block1_re… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_block1_bn1   │ (None, 56, 56,    │        256 │ stage2_block1_co… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_block1_relu1 │ (None, 56, 56,    │          0 │ stage2_block1_bn… │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_block1_conv2 │ (None, 56, 56,    │     36,864 │ stage2_block1_re… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_block1_proj… │ (None, 56, 56,    │      2,048 │ stage1_block1_re… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_block1_bn2   │ (None, 56, 56,    │        256 │ stage2_block1_co

 Total params: 2,258,209 (8.61 MB)

 Trainable params: 751,989 (2.87 MB)

 Non-trainable params: 2,240 (8.75 KB)

 Optimizer params: 1,503,980 (5.74 MB)

I0000 00:00:1777468838.677751 12975996 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Round-trip OK: reloaded model produces identical raw outputs.


## Validation sample overlays


In [5]:
dataset = FreiHand()
dataset.validate()
_, val_idx = dataset.train_validation_split(
    validation_fraction=SPLIT_VALIDATION_FRACTION,
    seed=SPLIT_SEED,
)

sample_ids = val_idx[:6]
images, ground_truth = dataset.load_batch(
    sample_ids,
    image_size=(input_size, input_size),
    flatten_keypoints=False,
)
predictions = eval_model.predict(images, verbose=0).reshape(-1, 21, 2)

fig = prediction_grid(
    images,
    ground_truth,
    predictions,
    ncols=3,
    suptitle=f'{IMPROVED_RUN}: validation predictions',
)
plt.show()


/var/folders/6x/nj8_bs6j4yq01x8v0rvq1gdr0000gn/T/ipykernel_65886/2058706548.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Full validation metric


In [6]:
val_ds = dataset.tf_dataset(
    indices=val_idx,
    batch_size=64,
    image_size=(input_size, input_size),
    flatten_keypoints=True,
)
metrics = evaluate_model(eval_model, val_ds, include_representative_indices=True)
print(metrics)


{'mpke_px': 8.482645034790039, 'median_sample_mpke_px': 7.251946449279785, 'p75_sample_mpke_px': 10.398750305175781, 'p90_sample_mpke_px': 14.272793769836426, 'p95_sample_mpke_px': 17.769163131713867, 'max_sample_mpke_px': 53.668060302734375, 'representative_indices': {'best': 1394, 'median': 885, 'p90': 1803, 'worst': 2984}, 'n_samples': 3256}


## Representative samples


In [7]:
if evaluation_path.exists():
    evaluation = json.loads(evaluation_path.read_text())
    representative = evaluation['metrics']['representative_indices']
else:
    representative = metrics['representative_indices']

labels = ['best', 'median', 'p90', 'worst']
sample_ids = np.array([val_idx[representative[label]] for label in labels])
images, ground_truth = dataset.load_batch(
    sample_ids,
    image_size=(input_size, input_size),
    flatten_keypoints=False,
)
predictions = eval_model.predict(images, verbose=0).reshape(-1, 21, 2)
errors = sample_mpke(predictions, ground_truth)
titles = [f'{label} ({error:.1f} px)' for label, error in zip(labels, errors)]

fig = prediction_grid(
    images,
    ground_truth,
    predictions,
    titles=titles,
    ncols=2,
    suptitle=f'{IMPROVED_RUN}: representative predictions',
)
saved_path = save_figure(fig, f'{IMPROVED_RUN}_representative_predictions')
print(f'Saved figure: {saved_path.relative_to(project_root)}')
plt.show()


Saved figure: reports/figures/improved-model-1_representative_predictions.png


/var/folders/6x/nj8_bs6j4yq01x8v0rvq1gdr0000gn/T/ipykernel_65886/595158457.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
